In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import torch
import numpy as np
import yfinance as yf
import pandas as pd
from operator import itemgetter
from models.lstm_model import LSTM_Model
from utils import get_device, build_seq, plus_bus_day, Get_yf_data

# Parameter

In [3]:
ckpt_path = '../checkpoints/cnn_checkpoint.pt'
ticker = 'VOO'
from_date = '2026-01-01'

# Extract saved objects and configs

In [4]:
device = get_device()
checkpoint = torch.load("../checkpoints/lstm_checkpoint.pt", map_location = device)

In [5]:
model_config = checkpoint["model_config"]
x_scaler = checkpoint["x_scaler"]
y_scaler = checkpoint["y_scaler"]
input_cols = checkpoint["input_cols"]
seq_len = checkpoint["preprocess_config"]["seq_length"]
steps_ahead = checkpoint["steps_ahead"]
target_cols = checkpoint['target_cols']

# Rebuild model

In [6]:
model = LSTM_Model(**model_config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
_ = model.eval()

# Get Latest Data

In [8]:
data = Get_yf_data( ticker , '2026-01-01' )

[*********************100%***********************]  1 of 1 completed


# Prepare Data

In [10]:
X_latest = x_scaler.transform(data[input_cols].values) # .astype(np.float32)
y_latest = y_scaler.fit_transform(data[target_cols].values)

In [11]:
Date_all = data['Date'].values.flatten().tolist()
Seqs = build_seq( seq_len, X_latest, steps_ahead, Date_all, y_latest, True)
X_seq, X_Dates, y_Seq = itemgetter('X_Seq','X_id','y_Seq')(Seqs)

In [23]:
Result_list = []


for id, item in enumerate(X_Dates):
    
    fr = item[0].strftime('%Y-%m-%d')
    to = item[-1].strftime('%Y-%m-%d')

    actual = y_scaler.inverse_transform([y_Seq[id]])[0, 0] if y_Seq[id] is not None else None

    
    predict_day = plus_bus_day(to, steps_ahead)

    predict_seq = X_seq[id]
    Predict_Tensor = torch.tensor(predict_seq, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        
        pred_scaled = model(Predict_Tensor).cpu().numpy()    
        pred_real = y_scaler.inverse_transform(pred_scaled)[0, 0]


    result = {
                "Ref_From" : fr,
                "Ref_To" : to,
                "Date": predict_day,
                "Actual": round(actual,3)if actual is not None else None, 
                "Predict": round(pred_real,3),
                "Difference": pred_real - actual if actual is not None else None,       
             }
    
    Result_list.append(result)


In [24]:
# pd.DataFrame(Result_list).to_csv('test.csv')

In [25]:
display(pd.DataFrame(Result_list))

,Ref_From,Ref_To,Date,Actual,Predict,Difference
0,2026-01-02,2026-02-13,2026-02-23,625.655,656.119019,30.463562
1,2026-01-05,2026-02-17,2026-02-24,630.221,655.992981,25.772278
2,2026-01-06,2026-02-18,2026-02-25,635.524,655.896973,20.372803
3,2026-01-07,2026-02-19,2026-02-26,631.955,655.765991,23.810974
4,2026-01-08,2026-02-20,2026-02-27,629.054,655.650024,26.595581
5,2026-01-09,2026-02-23,2026-03-02,629.294,655.585999,26.291992
6,2026-01-12,2026-02-24,2026-03-03,623.751,655.507019,31.755920
7,2026-01-13,2026-02-25,2026-03-04,628.197,655.546997,27.350220
8,2026-01-14,2026-02-26,2026-03-05,624.838,655.614014,30.775940
9,2026-01-15,2026-02-27,2026-03-06,616.484,655.646973,39.162537
